In [ ]:
import xarray as xr
import cartopy as ctpy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime, timedelta
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.path as mpath
import matplotlib.patches as mpatches
import metpy.calc as mpcalc
import scipy.stats as stat
import seaborn as sns
import nctoolkit as nc
import pandas as pd
import dask.array as da
import xcdat
import importlib
import glob
from numpy import *
from metpy.calc import heat_index
from metpy.units import units
from netCDF4 import Dataset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import sys

import filepaths as filepaths
import model_preprocess as model_prep
import calculations as dr_g_calcs
import dictionary as dicts
importlib.reload(filepaths)
importlib.reload(model_prep)
importlib.reload(dr_g_calcs)
importlib.reload(dicts)
import math
import pickle

In [ ]:
yr_i=1981
yr_f=2005
var_choice='pr'
label_choice='pr_mm'
wrf_var='RAINNC'
plot_var='Precipitation'
timeframe='historical'
location_strs=['South Florida']
location_abbrevs=['South FL']
so_fl_ll=[25.,-82.]
so_fl_ur=[27.,-80.]

locations=[so_fl_ll,so_fl_ur]
lat_arr=array(locations)[:,0]
lon_arr=array(locations)[:,1]

open obs data and calculate corresponding cumulative anomaly curve

In [ ]:
%%time
files=filepaths.era5_filepath(var_choice,temporal='daily')
era5=model_prep.era5_open_file(files,var_choice,str(yr_i),str(yr_f))
era5.coords['lon'].attrs={'axis':"X"}
era5.coords['lat'].attrs={'axis':"Y"}
era5.coords['time'].attrs={'axis':"T"}
era5_plot=dr_g_calcs.cumulative_precip_anom_curve(era5,label_choice,lat_arr,lon_arr)
file_suffix=''
verification_label='ERA5'

load pickle and numerical files to quick load for plotting

In [ ]:
%%time
couple_list=load('coupled_'+str(timeframe)+'.npy').tolist()
uncouple_list=load('uncoupled_'+str(timeframe)+'.npy').tolist()
mesaclip_list=load('mesaclip_'+str(timeframe)+'.npy').tolist()
bccaq_list=load('bccaq_'+str(timeframe)+'.npy').tolist()
bcsd_list=load('gddp_'+str(timeframe)+'.npy').tolist()
loca_list=load('loca_'+str(timeframe)+'.npy').tolist()

with open('coupled_'+str(timeframe)+'.pkl', 'rb') as f:
    label_list_couple = pickle.load(f)

with open('uncoupled_'+str(timeframe)+'.pkl', 'rb') as f:
    label_list_uncouple = pickle.load(f)

with open('mesaclip_'+str(timeframe)+'.pkl', 'rb') as f:
    label_list_mesaclip = pickle.load(f)

with open('bccaq_'+str(timeframe)+'.pkl', 'rb') as f:
    label_list_bccaq = pickle.load(f)

with open('loca_'+str(timeframe)+'.pkl', 'rb') as f:
    label_list_loca = pickle.load(f)

with open('gddp_'+str(timeframe)+'.pkl', 'rb') as f:
    label_list_gddp = pickle.load(f)

consolidate mesaclip and coupled highiresmip into one dictionary

In [ ]:
ihesp=array(mesaclip_list)[0,:]
mesaclip_ens=nanmean(array(mesaclip_list),axis=0)
couple_list.append(ihesp)
couple_list.append(mesaclip_ens)
label_list_couple.append('iHESP')
label_list_couple.append('MESACLIP ENS MEAN')
label_list_couple[-1]='MESACLIP\nENS MEAN'

fix labeling to make figure more concise (supplemental only)

In [ ]:
label_list_loca[1]='HadGEM3 (MM)'
label_list_loca[2]='HadGEM3 (LL)'
label_list_gddp[2]='HadGEM3 (MM)'
label_list_gddp[3]='HadGEM3 (LL)'
label_list_bccaq[1]='HadGEM3 (LL)'

truncate along ensemble member axis to plot mean

In [ ]:
bccaq_ens=nanmean(array(bccaq_list),axis=0)
loca_ens=nanmean(array(loca_list),axis=0)
gddp_ens=nanmean(array(bcsd_list),axis=0)
uncouple_ens=nanmean(array(uncouple_list),axis=0)
couple_ens=nanmean(array(couple_list),axis=0)
mesaclip_ens=nanmean(array(mesaclip_list),axis=0)
mydata_list=[couple_ens,uncouple_ens,mesaclip_ens,bccaq_ens,loca_ens,gddp_ens]
label_list_mydata=['COUPLED MODEL MEAN','UNCOUPLED MODEL MEAN','MESACLIP ENS MEAN','BCCAQ MEAN','LOCA MEAN','NASA-NEX-GDDP MEAN']

supplemental (detailed, single ensemble member view)
VVVVVV

In [ ]:
%%time

if (timeframe=='historical'):
    ymins=-350
    ymaxs=250
    ytexts=287
    plt_time='Historical (1981-2005)'
else:
    ymins=-350
    ymaxs=250
    ytexts=262
    plt_time='SSP-5.85 (2015-2024)'

month_name=['J','F','M','A','M','J','J','A','S','O','N','D']
categories=['Coupled Models','Uncoupled Models','BCCAQ Downscaled Model Data','BCSD Downscaled Model Data','LOCA Downscaled Model Data']
data=[couple_list,uncouple_list,bccaq_list,bcsd_list,loca_list]
label_arr=[label_list_couple,label_list_uncouple,label_list_bccaq,label_list_gddp,label_list_loca]
color_stem=['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown','tab:pink','tab:olive','tab:cyan']
color_arr = mpl.color_sequences['tab10']

on_line=where(cumsum(era5_plot)==nanmin(cumsum(era5_plot)))
on_point=nanmin(cumsum(era5_plot))
dem_line=where(cumsum(era5_plot)==nanmax(cumsum(era5_plot)))
dem_point=nanmax(cumsum(era5_plot))

fig = plt.figure(figsize=(65, 30))
# Define GridSpec: 2 rows, 6 columns for flexible positioning
gs = gridspec.GridSpec(2, 6, figure=fig, 
                       height_ratios=[1, 1],      # Equal height rows
                       hspace=1,                # Vertical spacing between rows
                       wspace=6,                 # Horizontal spacing between columns
                       left=0.08, right=0.95,      # Outer margins
                       bottom=0.08, top=0.94)

# Top row: 2 centered panels (each spans 2 columns with 1-column gap on sides)
ax1 = fig.add_subplot(gs[0, 1:3])    # Columns 1-2 (centered-left)
ax2 = fig.add_subplot(gs[0, 3:5])    # Columns 3-4 (centered-right)

# Bottom row: 3 equal panels (each spans 2 columns)
ax3 = fig.add_subplot(gs[1, 0:2])    # Columns 0-1 (left)
ax4 = fig.add_subplot(gs[1, 2:4])    # Columns 2-3 (middle)
ax5 = fig.add_subplot(gs[1, 4:6])    # Columns 4-5 (right)

axes = [ax1, ax2, ax3, ax4, ax5]

count=0
for ax, dataval in zip(axes, data):

    on_times=[]
    dem_times=[]
    on_points=[]
    dem_points=[]
 
    ax_histx = ax.inset_axes([0, 1.07, 1, 0.55], sharex=ax) #x0,y0,w,h
    ax_histy = ax.inset_axes([1.02, 0, 0.55, 1], sharey=ax)
    for i in range(0,len(dataval)):
        on_val=nanmin(cumsum(dataval[i]))
        on_times.append(where(cumsum(dataval[i])==on_val))
        on_points.append(on_val)
        dem_val=nanmax(cumsum(dataval[i]))
        dem_times.append(where(cumsum(dataval[i])==dem_val))
        dem_points.append(dem_val)
    
    ax.plot(arange(1,367,1),cumsum(era5_plot),lw=10,ls='-',color='black',zorder=5,label=verification_label+' Data')#,alpha=0.6)
    
    for i in range(0,len(dataval)):
        ax.plot(arange(1,367,1),cumsum(dataval[i]),lw=5,ls='-',color=color_arr[i],zorder=1)
        
    ax.axvline(x=on_line[0],ls='-.',color='grey',lw=5,clip_on=False,zorder=0,label=verification_label+' Wet\nSeason Onset')
    ax.axvline(x=dem_line[0],ls='--',color='grey',lw=5,clip_on=False,zorder=0,label=verification_label+' Wet\nSeason Demise')
    ax.axhline(y=0,ls='-',color='black',lw=5,zorder=0)
    
    if (count==0) | (count==2):
        ax.set_ylabel('\nAnomalous Accumulation (mm)\n',size=30,fontweight='bold')
    
    ax.set_yticks(arange(-350,300,50))
    ax.tick_params(axis='y',labelsize=25)
    plt.setp(ax.get_yticklabels(), fontweight='bold')
    
    ax_histx.axvline(x=on_line[0],ls='-.',color='grey',lw=5,clip_on=False,ymin=-0.2,ymax=1)
    ax_histx.axvline(x=dem_line[0],ls='--',color='grey',lw=5,clip_on=False,ymin=-0.2,ymax=1)

    spacing=linspace(0.1,0.9,len(dataval))
    nums=[]
    for i in range(0,len(dataval)):
        ax_histx.plot(on_times[i][0], spacing[i],marker='o',markersize=60,markeredgewidth=5,markeredgecolor=color_arr[i],markerfacecolor='white',zorder=2)#,fillstyle='none')#,linewidth=50)
        ax_histx.plot(dem_times[i][0], spacing[i],marker='o',markersize=60,markeredgewidth=5,markeredgecolor=color_arr[i],markerfacecolor='white',zorder=2)#,fillstyle='none')#,linewidth=50)
        ax_histx.plot([on_times[i][0], dem_times[i][0]], [spacing[i],spacing[i]], linestyle='-', color=color_arr[i],lw=5,zorder=0)
        txt1=on_line[0]-on_times[i][0]
        txt2=dem_times[i][0]-dem_line[0]
        if (txt1 > 0):
            txt_plot1='+'+str(txt1[0])
        else:
            txt_plot1=str(txt1[0])

        if (txt2 > 0):
            txt_plot2='+'+str(txt2[0])
        else:
            txt_plot2=str(txt2[0])
            
        ax_histx.text(on_times[i][0], spacing[i], str(txt_plot1), ha='center', va='center',size=25,fontweight='bold')
        ax_histx.text(dem_times[i][0], spacing[i], str(txt_plot2), ha='center', va='center',size=25,fontweight='bold')
        
        nums.append(label_arr[count][i])
    ax_histx.set_xlabel('')
    ax_histx.set_ylabel('')
   
    ax_histx.tick_params(bottom=False,labelbottom=False)
    ax_histx.set_yticks(spacing,labels=nums,size=30,ha='left')
    ax_histx.tick_params(axis='y', direction='in',pad=-25)
    ax_histx.set_ylim(0,1)
    ax.set_xlim(1,366)
    ax.set_xticks([1,32,60,91,121,152,182,213,244,274,305,335],labels=month_name,fontsize=30,fontweight='bold')
    ax.set_ylim(ymins,ymaxs)
    ax.legend(loc='upper left',fontsize=25)
    ax_histx.set_title(categories[count]+'\n',size=30,fontweight='bold')
    ax_histx.fill_betweenx(arange(0,1.1,0.1),on_line[0], dem_line[0],color='gray', alpha=0.1)

    ax_histy.axhline(y=0,ls='-',color='black',lw=2)
    if count > 2:
        spacing=linspace(0.07,0.93,len(dataval))
    else:
        spacing=linspace(0.1,0.9,len(dataval))
    for i in range(0,len(dataval)):
        markerline,a,b=ax_histy.stem(spacing[i],on_points[i],markerfmt='s', bottom=on_point, orientation='vertical',linefmt=color_stem[i])
        markerline.set_markersize(60) #19
        markerline.set_markeredgewidth(4)
        markerline.set_markeredgecolor(color_stem[i])
        markerline.set_markerfacecolor('white')
        a.set_linewidth(4)
        markerline.set_zorder(3)
        markerline,a,b=ax_histy.stem(spacing[i],dem_points[i],markerfmt='s', bottom=dem_point, orientation='vertical',linefmt=color_stem[i])
        markerline.set_markersize(60)
        markerline.set_markeredgecolor(color_stem[i])
        markerline.set_markerfacecolor('white')
        a.set_linewidth(4)
        markerline.set_markeredgewidth(4) 
        markerline.set_zorder(3)

        txt1=round(on_point-on_points[i])
        txt2=round(dem_points[i]-dem_point)
        if (on_points[i] < on_point):
            txt_plot1='+'+str(txt1)
        else:
            txt_plot1=str(txt1)

        if (dem_points[i] >= dem_point):
            txt_plot2='+'+str(txt2)
        else:
            txt_plot2=str(txt2)
            
        ax_histy.text(spacing[i],on_points[i],str(txt_plot1)+'\nmm', ha='center', va='center',size=25,fontweight='bold')
        ax_histy.text(spacing[i],dem_points[i],str(txt_plot2)+'\nmm', ha='center', va='center',size=25,fontweight='bold')
    ax_histy.tick_params(left=False,labelleft=False,bottom=False,labelbottom=False)
    
    ax_histy.tick_params(axis='y', direction='in',pad=-10)
    ax_histy.set_xlim(0,1)
    ax_histy.fill_between(arange(0,1.1,0.1), on_point, dem_point, color='gray', alpha=0.1)
    ax_histy.axhline(y=round(on_point),clip_on=False,xmin=-0.12,xmax=1.,color='black',linewidth=3)
    ax_histy.axhline(y=round(dem_point),clip_on=False,xmin=-0.12,xmax=1.,color='black',linewidth=3)
   
    ax_twin = ax.twinx()
    ax_twin.set_ylim(ax.get_ylim())
    ax_twin.set_yticks([on_point,dem_point],labels=[str(round(on_point))+'\nmm\n\n\n',str(round(dem_point))+'\nmm\n\n\n'],ha='right',fontweight='bold')
    ax_twin.tick_params(axis='y', direction='in',pad=-10,labelsize=30)
    
    count+=1
    
mpl.rcParams['font.family'] = "sans-serif"
mpl.rcParams['font.sans-serif'] = "Verdana" 
plt.rcParams['savefig.dpi'] = 300
plt.rc('xtick.major', width=2, size=20)
plt.rc('ytick.major', width=2, size=20)
mpl.rcParams['axes.linewidth'] = 4 


simplified (ens mean) view (figs 2-3) VVVVVV

In [ ]:
%%time
if (timeframe=='historical'):
    ymins=-350
    ymaxs=250
    ytexts=287
    plt_time='Historical (1981-2005)'
else:
    ymins=-350
    ymaxs=250
    ytexts=262
    plt_time='SSP-5.85 (2015-2024)'

month_name=['J','F','M','A','M','J','J','A','S','O','N','D']

categories=['Coupled Models','Uncoupled Models','Downscaled Data']
color_stem=['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown','tab:pink','tab:olive','tab:cyan']
color_arr = mpl.color_sequences['tab10']

on_line=where(cumsum(era5_plot)==nanmin(cumsum(era5_plot)))
on_point=nanmin(cumsum(era5_plot))
dem_line=where(cumsum(era5_plot)==nanmax(cumsum(era5_plot)))
dem_point=nanmax(cumsum(era5_plot))

count=0
fig = plt.figure(figsize=(35, 30),layout='constrained')
ax = fig.add_subplot()

ax_histx = ax.inset_axes([0, 1.03, 1, 0.27], sharex=ax)
ax_histy = ax.inset_axes([1.02, 0, 0.25, 1], sharey=ax)

spacing=linspace(0.1,0.9,len(mydata_list))
ax.plot(arange(1,367,1),cumsum(era5_plot),lw=12,ls='-',color='black',zorder=5,label=verification_label+' Data')#,alpha=0.6)
ax.axvline(x=on_line,ls='-.',color='grey',lw=5,clip_on=False,zorder=0,label=verification_label+' Wet Season Onset')
ax.axvline(x=dem_line,ls='--',color='grey',lw=5,clip_on=False,zorder=0,label=verification_label+' Wet Season Demise')

for i,dataval in enumerate(mydata_list):

    on_val=nanmin(cumsum(dataval))
    on_times=where(cumsum(dataval)==on_val)
    dem_val=nanmax(cumsum(dataval))
    dem_times=where(cumsum(dataval)==dem_val)
    
    ax.plot(arange(1,367,1),cumsum(dataval),lw=7,ls='-',color=color_arr[i],zorder=1)
        
    ax.axhline(y=0,ls='-',color='black',lw=4,zorder=0)
    
    ax.set_ylabel('Anomalous Accumulation (mm)',size=25)
    ax.set_xlabel('\nMonth of Year',fontsize=25,fontweight='bold')
    ax.set_yticks(arange(ymins,ymaxs+50,50))
    ax.tick_params(axis='y', labelsize=25)
    ax.set_ylabel('\nAnomalous Accumulation (mm)\n',size=30,fontweight='bold')
    plt.setp(ax.get_yticklabels(), fontweight='bold')
    
    ax_histx.axvline(x=on_line,ls='-.',color='grey',lw=5,clip_on=False,ymin=-0.2,ymax=1,zorder=1)
    ax_histx.axvline(x=dem_line,ls='--',color='grey',lw=5,clip_on=False,ymin=-0.1,ymax=1,zorder=1)
    
    ax_histx.plot(on_times[0][0], spacing[i],marker='o',markersize=60,markeredgewidth=5,markeredgecolor=color_arr[i],markerfacecolor='white',zorder=3)#,fillstyle='none')#,linewidth=50)
    ax_histx.plot(dem_times[0][0], spacing[i],marker='o',markersize=60,markeredgewidth=5,markeredgecolor=color_arr[i],markerfacecolor='white',zorder=3)#,fillstyle='none')#,linewidth=50)
    ax_histx.plot([on_times[0][0], dem_times[0][0]], [spacing[i],spacing[i]], linestyle='-', color=color_arr[i],lw=5,zorder=2)
    txt1=on_line[0]-on_times[0][0]
    txt2=dem_times[0][0]-dem_line[0]
    if (txt1 > 0):
        txt_plot1='+'+str(txt1[0])
    else:
        txt_plot1=str(txt1[0])

    if (txt2 > 0):
        txt_plot2='+'+str(txt2[0])
    else:
        txt_plot2=str(txt2[0])
        
    ax_histx.text(on_times[0][0], spacing[i], str(txt_plot1), ha='center', va='center',size=25,fontweight='bold')
    ax_histx.text(dem_times[0][0], spacing[i], str(txt_plot2), ha='center', va='center',size=25,fontweight='bold')
    
    ax_histx.set_xlabel('')
    ax_histx.set_ylabel('')
  
    ax_histx.set_ylabel('\nWet Season Length Bias (days)',fontsize=20,fontweight='bold',rotation=270)#,labelpad=75)
    ax_histx.yaxis.set_label_position("right")
    
    ax_histx.tick_params(bottom=False,labelbottom=False)
    ax_histx.set_yticks(spacing,labels=label_list_mydata,size=25,ha='left',fontweight='bold')
    ax_histx.tick_params(axis='y', direction='in',pad=-20)
    ax_histx.set_ylim(0,1)
    ax.set_xlim(1,366)
    ax.set_xticks([1,32,60,91,121,152,182,213,244,274,305,335],labels=month_name,fontsize=25,fontweight='bold')
    ax.set_ylim(ymins,ymaxs)
    ax.legend(loc='best',prop={'weight': 'bold','size':25})
    
    ax_histx.fill_betweenx(arange(0,1.1,0.1),on_line[0], dem_line[0],color='gray', alpha=0.1)

    ax_histy.axhline(y=0,ls='-',color='black',lw=2)
   
    markerline,a,b=ax_histy.stem(spacing[i],on_val,markerfmt='s', bottom=on_point, orientation='vertical',linefmt=color_stem[i])
    markerline.set_markersize(60) #19
    markerline.set_markeredgewidth(5)
    markerline.set_markeredgecolor(color_stem[i])
    markerline.set_markerfacecolor('white')
    a.set_linewidth(5)
    markerline.set_zorder(3)
    markerline,a,b=ax_histy.stem(spacing[i],dem_val,markerfmt='s', bottom=dem_point, orientation='vertical',linefmt=color_stem[i])
    markerline.set_markersize(60)
    markerline.set_markeredgecolor(color_stem[i])
    markerline.set_markerfacecolor('white')
    a.set_linewidth(5)
    markerline.set_markeredgewidth(5) 
    markerline.set_zorder(3)

    txt1=round(on_point-on_val)
    txt2=round(dem_val-dem_point)
    if (on_val < on_point):
        txt_plot1='+'+str(txt1)
    else:
        txt_plot1=str(txt1)

    if (dem_val >= dem_point):
        txt_plot2='+'+str(txt2)
    else:
        txt_plot2=str(txt2)
        
    ax_histy.text(spacing[i],on_val,str(txt_plot1)+'\nmm', ha='center', va='center',size=25,fontweight='bold')
    ax_histy.text(spacing[i],dem_val,str(txt_plot2)+'\nmm', ha='center', va='center',size=25,fontweight='bold')
    ax_histy.tick_params(left=False,labelleft=False,bottom=False,labelbottom=False)
   
    ax_histy.tick_params(axis='y', direction='in',pad=-10)
    ax_histy.set_xlim(0,1)
    ax_histy.fill_between(arange(0,1.1,0.1), on_point, dem_point, color='gray', alpha=0.1)
    ax_histy.axhline(y=round(on_point),clip_on=False,xmin=-0.12,xmax=1.,color='black',linewidth=3)
    ax_histy.axhline(y=round(dem_point),clip_on=False,xmin=-0.12,xmax=1.,color='black',linewidth=3)
    ax_histy.set_xlabel('Rainfall Bias (mm)',fontsize=20,fontweight='bold',labelpad=-35)
    ax_histy.xaxis.set_label_position("top")
   
    ax_twin = ax.twinx()
    ax_twin.set_ylim(ax.get_ylim())
    ax_twin.set_yticks([on_point,dem_point],labels=[str(round(on_point))+' mm',str(round(dem_point))+' mm'],ha='right',fontweight='bold')
    ax_twin.tick_params(axis='y', direction='in',pad=-25,labelsize=30)

mpl.rcParams['font.family'] = "sans-serif"
mpl.rcParams['font.sans-serif'] = "Verdana" 
plt.rcParams['savefig.dpi'] = 300
plt.rc('xtick.major', width=2, size=20)
plt.rc('ytick.major', width=2, size=20)
mpl.rcParams['axes.linewidth'] = 4 

plt.suptitle('\n'+plt_time+' Cumulative Rainfall Deficit (mm/day)\nfor Southern Florida: '+f"25\u00B0N - 27\u00B0N, 82\u00B0W - 80\u00B0W\n",fontsize=25,fontweight='bold')